# DAT341 Programming Assignment 4
- PA 4 4 Group Members:
  
  1. Zheng Wei Liau (liau@chalmers.se)
  2. Jared Ng (jaredn@chalmers.se)

# Task 1: Analysing Dataset and Linear Classifiers


In [1]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import Perceptron
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score
from sklearn.pipeline import make_pipeline

X1 = [{'city':'Gothenburg', 'month':'July'},
      {'city':'Gothenburg', 'month':'December'},
      {'city':'Paris', 'month':'July'},
      {'city':'Paris', 'month':'December'}]
Y1 = ['rain', 'rain', 'sun', 'rain']

X2 = [{'city':'Sydney', 'month':'July'},
      {'city':'Sydney', 'month':'December'},
      {'city':'Paris', 'month':'July'},
      {'city':'Paris', 'month':'December'}]
Y2 = ['rain', 'sun', 'sun', 'rain']

classifier1 = make_pipeline(DictVectorizer(), Perceptron(max_iter=10))
classifier1.fit(X1, Y1)
guesses1 = classifier1.predict(X1)
print(accuracy_score(Y1, guesses1))

classifier2 = make_pipeline(DictVectorizer(), Perceptron(max_iter=10))
#classifier2 = make_pipeline(DictVectorizer(), LinearSVC())
classifier2.fit(X2, Y2)
guesses2 = classifier2.predict(X2)
print(accuracy_score(Y2, guesses2))


1.0
0.5


# Questions to answer:
- Why can't we improve the accuracy by switching to a LinearSVC?
- Why could the classifier "memorize" the training data in the first case, but not in the second case?



-- We cannot improve on the accuracy by switching to LinearSVC because the second dataset is not linearly separable unlike the first.

-- Linearly separable means that a stright line can be drawn which perfectly separates the two classes of 'sun' and 'rain'

-- In the second dataset a XOR pattern is found whereby neither of the features of 'city' or 'month' alone are able to separate the classes

-- Also, it is worth noting that both LinearSVC and Perceptrons are linear models so that require a dataset to be linearly separable to perform well.

# Task 2: Preparation for next tasks
- Understand how the Python code corresponds to the pseudocode in the lecture
- Next two code cells are given code
- Run the two code cells and make sure it works with an accuracy of about 0.8 obtained


In [2]:
"""This file shows a couple of implementations of the perceptron learning
algorithm. It is based on the code from Lecture 3, but using the slightly
more compact perceptron formulation that we saw in Lecture 6.

There are two versions: Perceptron, which uses normal NumPy vectors and
matrices, and SparsePerceptron, which uses sparse vectors and matrices.
The latter may be faster when we have high-dimensional feature representations
with a lot of zeros, such as when we are using a "bag of words" representation
of documents.
"""

import numpy as np
from sklearn.base import BaseEstimator

class LinearClassifier(BaseEstimator):
    """
    General class for binary linear classifiers. Implements the predict
    function, which is the same for all binary linear classifiers. There are
    also two utility functions.
    """

    def decision_function(self, X):
        """
        Computes the decision function for the inputs X. The inputs are assumed to be
        stored in a matrix, where each row contains the features for one
        instance.
        """
        return X.dot(self.w)

    def predict(self, X):
        """
        Predicts the outputs for the inputs X. The inputs are assumed to be
        stored in a matrix, where each row contains the features for one
        instance.
        """

        # First compute the output scores
        scores = self.decision_function(X)

        # Select the positive or negative class label, depending on whether
        # the score was positive or negative.
        out = [self.positive_class if score >= 0.0 else self.negative_class for score in scores]

        #### np.select([scores >= 0.0, scores < 0.0],
        ####                [self.positive_class,
        ####                 self.negative_class])
        return out

    def find_classes(self, Y):
        """
        Finds the set of output classes in the output part Y of the training set.
        If there are exactly two classes, one of them is associated to positive
        classifier scores, the other one to negative scores. If the number of
        classes is not 2, an error is raised.
        """
        classes = sorted(set(Y))
        if len(classes) != 2:
            raise Exception("this does not seem to be a 2-class problem")
        self.positive_class = classes[1]
        self.negative_class = classes[0]

    def encode_outputs(self, Y):
        """
        A helper function that converts all outputs to +1 or -1.
        """
        return np.array([1 if y == self.positive_class else -1 for y in Y])


class Perceptron(LinearClassifier):
    """
    A straightforward implementation of the perceptron learning algorithm.
    """

    def __init__(self, n_iter=20):
        """
        The constructor can optionally take a parameter n_iter specifying how
        many times we want to iterate through the training set.
        """
        self.n_iter = n_iter

    def fit(self, X, Y):
        """
        Train a linear classifier using the perceptron learning algorithm.
        """

        # First determine which output class will be associated with positive
        # and negative scores, respectively.
        self.find_classes(Y)

        # Convert all outputs to +1 (for the positive class) or -1 (negative).
        Ye = self.encode_outputs(Y)

        # If necessary, convert the sparse matrix returned by a vectorizer
        # into a normal NumPy matrix.
        if not isinstance(X, np.ndarray):
            X = X.toarray()

        # Initialize the weight vector to all zeros.
        n_features = X.shape[1]
        self.w = np.zeros(n_features)

        # Perceptron algorithm:
        for i in range(self.n_iter):
            for x, y in zip(X, Ye):

                # Compute the output score for this instance.
                score = x.dot(self.w)

                # If there was an error, update the weights.
                if y*score <= 0:
                    self.w += y*x


##### The following part is for the optional task.

### Sparse and dense vectors don't collaborate very well in NumPy/SciPy.
### Here are two utility functions that help us carry out some vector
### operations that we'll need.

def add_sparse_to_dense(x, w, factor):
    """
    Adds a sparse vector x, scaled by some factor, to a dense vector.
    This can be seen as the equivalent of w += factor * x when x is a dense
    vector.
    """
    w[x.indices] += factor * x.data

def sparse_dense_dot(x, w):
    """
    Computes the dot product between a sparse vector x and a dense vector w.
    """
    return np.dot(w[x.indices], x.data)


class SparsePerceptron(LinearClassifier):
    """
    A straightforward implementation of the perceptron learning algorithm,
    assuming that the input feature matrix X is sparse.
    """

    def __init__(self, n_iter=20):
        """
        The constructor can optionally take a parameter n_iter specifying how
        many times we want to iterate through the training set.
        """
        self.n_iter = n_iter

    def fit(self, X, Y):
        """
        Train a linear classifier using the perceptron learning algorithm.

        Note that this will only work if X is a sparse matrix, such as the
        output of a scikit-learn vectorizer.
        """
        self.find_classes(Y)

        # First determine which output class will be associated with positive
        # and negative scores, respectively.
        Ye = self.encode_outputs(Y)

        # Initialize the weight vector to all zeros.
        self.w = np.zeros(X.shape[1])

        # Iteration through sparse matrices can be a bit slow, so we first
        # prepare this list to speed up iteration.
        XY = list(zip(X, Ye))

        for i in range(self.n_iter):
            for x, y in XY:

                # Compute the output score for this instance.
                # (This corresponds to score = x.dot(self.w) above.)
                score = sparse_dense_dot(x, self.w)

                # If there was an error, update the weights.
                if y*score <= 0:
                    # (This corresponds to self.w += y*x above.)
                    add_sparse_to_dense(x, self.w, y)

In [3]:
print(Perceptron.__module__)

__main__


In [4]:
import time

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import Normalizer
from sklearn.pipeline import make_pipeline
from sklearn.feature_selection import SelectKBest
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

#from aml_perceptron import Perceptron, SparsePerceptron

# This function reads the corpus, returns a list of documents, and a list
# of their corresponding polarity labels.
def read_data(corpus_file):
    X = []
    Y = []
    with open(corpus_file, encoding='utf-8') as f:
        for line in f:
            _, y, _, x = line.split(maxsplit=3)
            X.append(x.strip())
            Y.append(y)
    return X, Y


if __name__ == '__main__':

    # Read all the documents.
    X, Y = read_data('all_sentiment_shuffled.txt')

    # Split into training and test parts.
    Xtrain, Xtest, Ytrain, Ytest = train_test_split(X, Y, test_size=0.2,
                                                    random_state=0)

    # Set up the preprocessing steps and the classifier.
    pipeline = make_pipeline(
        TfidfVectorizer(),
        SelectKBest(k=1000),
        Normalizer(),

        # NB that this is our Perceptron, not sklearn.linear_model.Perceptron
        Perceptron()
    )

    # Train the classifier.
    t0 = time.time()
    pipeline.fit(Xtrain, Ytrain)
    t1 = time.time()
    print('Training time: {:.2f} sec.'.format(t1-t0))

    # Evaluate on the test set.
    Yguess = pipeline.predict(Xtest)
    print('Accuracy: {:.4f}.'.format(accuracy_score(Ytest, Yguess)))

Training time: 1.65 sec.
Accuracy: 0.7919.


/usr/local/lib/python3.11/dist-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(


- Using the Perceptron, a test accuracy of 79.19% (4sf) is obtained.

# Task 3: Implementing the SVC
Task Requirements:
- Implement Pegasos algorithm for training Support Vector Classifiers using the pseudocode in the image above
- Test the Implementation with the above testing code
- HINT: Make a copy of the class Perceptron and then modify it to implement the Pegasos Algorithm

- Find good values for:
  a) Regularisation parameter
  b) Number of training steps ie. Number of iterations through the training set, or Number of selected training instances
- Accuracy should be >= 0.8






In [5]:
class PegasosSVM(LinearClassifier):
  def __init__(self, lambda_param=0.0001, n_iter=1000):
    self.lambda_param = lambda_param
    self.n_iter = n_iter

  def fit(self, X, Y):
    self.find_classes(Y)
    Ye = self.encode_outputs(Y)

    if not isinstance(X, np.ndarray):
      X = X.toarray()

    n_samples, n_features = X.shape
    self.w = np.zeros(n_features)

    for t in range(1, self.n_iter+1):
      i = np.random.randint(0, n_samples)
      x_i = X[i]
      y_i = Ye[i]

      eta_t = 1 / (self.lambda_param*t)

      if y_i * np.dot(self.w, x_i) < 1:
        self.w = (1 - eta_t * self.lambda_param) * self.w + eta_t * y_i * x_i
      else:
        self.w = (1 - eta_t * self.lambda_param) * self.w

      # Optional projection step
      norm_w = np.linalg.norm(self.w)
      if norm_w != 0:
        projection_factor = min(1, 1 / (norm_w * np.sqrt(self.lambda_param)))
        self.w = projection_factor * self.w

    return self

  def predict(self, X):
    return super().predict(X)


In [6]:
import time

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import Normalizer
from sklearn.pipeline import make_pipeline
from sklearn.feature_selection import SelectKBest
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split, GridSearchCV

#from aml_perceptron import Perceptron, SparsePerceptron

# This function reads the corpus, returns a list of documents, and a list
# of their corresponding polarity labels.
def read_data(corpus_file):
    X = []
    Y = []
    with open(corpus_file, encoding='utf-8') as f:
        for line in f:
            _, y, _, x = line.split(maxsplit=3)
            X.append(x.strip())
            Y.append(y)
    return X, Y


if __name__ == '__main__':

    # Read all the documents.
    X, Y = read_data('all_sentiment_shuffled.txt')

    # Split into training and test parts.
    Xtrain, Xtest, Ytrain, Ytest = train_test_split(X, Y, test_size=0.2,
                                                    random_state=0)

    # Set up the preprocessing steps and the classifier.
    pipeline = make_pipeline(
        TfidfVectorizer(),
        SelectKBest(k=1000),
        Normalizer(),

        # Changed to Pegasos SVM
        PegasosSVM()
    )

    param_grid = {
        'pegasossvm__lambda_param': [0.00001, 0.0001, 0.001, 0.01, 0.1],
        'pegasossvm__n_iter': [1000, 5000, 10000]
    }

    grid = GridSearchCV(pipeline, param_grid, scoring='accuracy', cv=5, n_jobs=-1, verbose=2)

    # Train the classifier.
    t0 = time.time()
    grid.fit(Xtrain, Ytrain)
    t1 = time.time()
    print('Training time: {:.2f} sec.'.format(t1-t0))

    # Evaluate on the test set.
    print('Best parameters: {}'.format(grid.best_params_))
    print('Best cross-validation accuracy: {:.4f}.'.format(grid.best_score_))

    Yguess = grid.best_estimator_.predict(Xtest)
    print('Test Accuracy: {:.4f}.'.format(accuracy_score(Ytest, Yguess)))

Fitting 5 folds for each of 15 candidates, totalling 75 fits
Training time: 96.89 sec.
Best parameters: {'pegasossvm__lambda_param': 0.001, 'pegasossvm__n_iter': 10000}
Best cross-validation accuracy: 0.8184.
Test Accuracy: 0.8082.


/usr/local/lib/python3.11/dist-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(


- Using the Pegasos Algorithm with Hinge Loss to implement the Support Vector Classifier (SVC), we obtained a test accuracy of 80.82% and the best parameters for the SVC is when the regularisation parameter is 0.001 and when the number of iterations is 10000.

# Task 4: Logistic Regression
We will be using Log Loss instead of Hinge Loss, we will be using the formula below:

- $ Loss(w, x_i, y_i) = log(1+exp(y_i . (w.x_i))) $

- $ ∇(Loss) = -y_i / (1 + exp(y_i . (w . x_i))) . x_i$

- $ ∇(f) = λ . w + ∇(Loss) $


Task Requirements:
- Similar to SVC but uses Loss Function instead of Hinge Loss
- Read through explanation of log loss and its gradient in section 3 in the clarification document, OR look at the table on page 15 in the Pegasos paper
  
  - First line shows the Hinge Loss and its gradient (Hinge Loss does not have a gradient at 1, is a subgradient). This difference does not matter in this course.
  - Second line shows Log Loss and the corresponding gradient
- Code a new training algorithm that uses Log Loss instead of Hinge Loss
- Describe how well it works compared to your previous classifier

In [7]:
class PegasosLogLoss(LinearClassifier):
  def __init__(self, lambda_param=0.001, n_iter=1000):
    self.lambda_param = lambda_param
    self.n_iter = n_iter

  def fit(self, X, Y):
    self.find_classes(Y)
    Ye = self.encode_outputs(Y)

    if not isinstance(X, np.ndarray):
      X = X.toarray()

    n_samples, n_features = X.shape
    self.w = np.zeros(n_features)

    for t in range(1, self.n_iter+1):
      i = np.random.randint(0, n_samples)
      x_i = X[i]
      y_i = Ye[i]

      eta_t = 1 / (self.lambda_param*t)

      loss = np.log(1 + np.exp(-y_i * np.dot(self.w, x_i)))
      gradient_of_loss = (-y_i * x_i) / (1 + np.exp(y_i * np.dot(self.w, x_i)))
      gradient_of_SVC_objective = self.lambda_param * self.w + gradient_of_loss

      self.w = self.w - (eta_t * gradient_of_SVC_objective)

      if(t == self.n_iter):
        print("Iteration: {}, Loss: {}, Lambda: {}".format(self.n_iter, loss, self.lambda_param))
      elif(t % 100 == 0):
        print("Iteration: {}, Loss: {}".format(t, loss))

    return self

  def predict(self, X):
    return super().predict(X)


In [8]:
def read_data(corpus_file):
    X = []
    Y = []
    with open(corpus_file, encoding='utf-8') as f:
        for line in f:
            _, y, _, x = line.split(maxsplit=3)
            X.append(x.strip())
            Y.append(y)
    return X, Y


if __name__ == '__main__':

    # Read all the documents.
    X, Y = read_data('all_sentiment_shuffled.txt')

    # Split into training and test parts.
    Xtrain, Xtest, Ytrain, Ytest = train_test_split(X, Y, test_size=0.2,
                                                    random_state=0)

    # Set up the preprocessing steps and the classifier.
    pipeline = make_pipeline(
        TfidfVectorizer(),
        SelectKBest(k=1000),
        Normalizer(),

        # Changed to Pegasos Log Loss
        PegasosLogLoss()
    )

    param_grid = {
        'pegasoslogloss__lambda_param': [0.00001, 0.0001, 0.001, 0.01, 0.1],
        'pegasoslogloss__n_iter': [1000, 5000, 10000]
    }

    grid = GridSearchCV(pipeline, param_grid, scoring='accuracy', cv=5, n_jobs=-1, verbose=2)

    # Train the classifier.
    t0 = time.time()
    grid.fit(Xtrain, Ytrain)
    t1 = time.time()
    print('Training time: {:.2f} sec.'.format(t1-t0))

    # Evaluate on the test set.
    print('Best parameters: {}'.format(grid.best_params_))
    print('Best cross-validation accuracy: {:.4f}.'.format(grid.best_score_))

    Yguess = grid.best_estimator_.predict(Xtest)
    print('Test Accuracy: {:.4f}.'.format(accuracy_score(Ytest, Yguess)))

Fitting 5 folds for each of 15 candidates, totalling 75 fits
Iteration: 100, Loss: 4.9855622482273
Iteration: 200, Loss: 1.0552558826202867e-10
Iteration: 300, Loss: 1.3811812930204898e-05
Iteration: 400, Loss: 9.719099608095165
Iteration: 500, Loss: 0.003170583772415458
Iteration: 600, Loss: 0.19123173576295624
Iteration: 700, Loss: 0.003668235915184702
Iteration: 800, Loss: 0.0013444685547335923
Iteration: 900, Loss: 3.8975971421107882
Iteration: 1000, Loss: 7.857318212569324e-06
Iteration: 1100, Loss: 0.1992444869049939
Iteration: 1200, Loss: 0.008653653868558757
Iteration: 1300, Loss: 0.0008887453435853527
Iteration: 1400, Loss: 0.00767654256759633
Iteration: 1500, Loss: 0.0005062155503338168
Iteration: 1600, Loss: 0.370823832664843
Iteration: 1700, Loss: 0.8522222816813625
Iteration: 1800, Loss: 0.0006327686309861718
Iteration: 1900, Loss: 1.1243389471249803
Iteration: 2000, Loss: 0.020447456433255858
Iteration: 2100, Loss: 0.9578829634700056
Iteration: 2200, Loss: 0.7499303445782

/usr/local/lib/python3.11/dist-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(


- Using the Pegasos Algorithmn with Log Loss to implement the Support Vector Classifier (SVC), we obtained a test accuracy of 82.54% and the best parameters for the SVC is when the regularisation parameter is 0.0001 and when the number of iterations is 10000.

The Pegasos Algorithmn with Log Loss achieved a higher accuracy as compared to the Pegasos Algorithmn with Hinge Loss. The Algorithm with Hinge Loss had an accuracy of 80.82% while the Algorithm with Log Loss had an accuracy of 82.54%. The Algorithmn with the Log Loss had a best parameter when the regularisation parameter is 0.0001 and number of iterations is 10000 while the Algorithmn with the Hinge Loss had a best paramter when the regularisation parameter is 0.001 and number of iterations is 10000.